In [ ]:
from pcamarillor.spark_utils import SparkUtils
from pyspark.sql.functions import col, from_json, when
from pyspark.sql.types import *

# Configuración de paquetes para Spark 4.0.1 (Scala 2.13)
packages = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0",
    "org.mongodb.spark:mongo-spark-connector_2.13:10.4.0"
])

su = SparkUtils(
    "StreamingApp",
    "spark://spark-master:7077",
    spark_packages=packages
)
spark = su.spark
print("Spark version:", spark.version)

# 1. Leer desde Kafka
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9093") \
    .option("subscribe", "videogames") \
    .option("startingOffsets", "latest") \
    .load()

# 2. Definir el esquema (Debe coincidir con el JSON que envía el Productor)
schema = StructType([
    StructField("id", StringType()),
    StructField("game", StringType()),
    StructField("genre", StringType()),
    StructField("platform", StringType()),
    StructField("price", DoubleType()),
    StructField("timestamp", TimestampType())
])

# 3. Parsear y agregar una transformación (Complejidad)
df_parsed = df_raw.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("price_category", 
        when(col("price") > 50, "Premium")
        .when(col("price") > 20, "Standard")
        .otherwise("Budget")
    )

# 4. Función foreachBatch para persistir en MongoDB (Requisito de 30 pts)
def write_to_mongo(batch_df, batch_id):
    # Condición de seguridad: solo escribir si el micro-batch no está vacío
    if not batch_df.isEmpty():
        batch_df.write \
            .format("mongodb") \
            .mode("append") \
            .option("database", "streaming_db") \
            .option("collection", "videogames") \
            .option("connection.uri", "mongodb://host.docker.internal:27017") \
            .option("writeConcern.w", "majority") \
            .option("operationType", "update") \
            .option("idFieldList", "id") \
            .save()

# 5. Iniciar el Streaming
print("Iniciando el streaming hacia MongoDB...")
query_mongo = df_parsed.writeStream \
    .foreachBatch(write_to_mongo) \
    .option("checkpointLocation", "/tmp/spark_checkpoints_v2") \
    .outputMode("update") \
    .start()



Spark version: 4.0.1
Iniciando el streaming hacia MongoDB...


26/05/08 15:37:22 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/08 15:37:26 WARN CaseInsensitiveStringMap: Converting duplicated key idfieldlist into CaseInsensitiveStringMap.
26/05/08 15:37:26 WARN CaseInsensitiveStringMap: Converting duplicated key operationType into CaseInsensitiveStringMap.
26/05/08 15:37:26 WARN CaseInsensitiveStringMap: Converting duplicated key writeConcern.w into CaseInsensitiveStringMap.
26/05/08 15:37:30 WARN CaseInsensitiveStringMap: Converting duplicated key idfieldlist into CaseInsensitiveStringMap.
26/05/08 15:37:30 WARN CaseInsensitiveStringMap: Converting duplicated key operationType into CaseInsensitiveStringMap.
26/05/08 15:37:30 WARN CaseInsensitiveStringMap: Converting duplicated key writeConcern.w into CaseInsensitiveStringMap.
26/05/08 15:37:31 WARN CaseInsensitiveStringMap: Converting duplicated key idfieldlist into CaseInsensitiveStringMap.
26/05/08 15:37:31 WA